# Argentina and Peru: source search to original reports

Both routes are experimental and need no API key. Install OpenCorpFilings with the `pdf` extra. These company pools use market-specific `search_companies()`, rather than the cross-market/browser directory. Results below were captured from live sources in October 2026; newer filings may be selected later.


In [1]:
from pathlib import Path
from opencorpfilings import Company, search_companies, provider_info

destination = Path("downloads/argentina-peru")
destination.mkdir(parents=True, exist_ok=True)
for market in ("AR", "PE"):
    profile = provider_info(market)
    print(market, profile.tier, profile.identifiers, profile.order_basis)


AR experimental ('CUIT', 'exact company name') filing_date
PE experimental ('SMV issuer selector ID', 'exact company name') report_year


## Argentina: CUIT, submission UUID and original PDFs

CNV lists financial-statement submissions with a filing date and explicit closing date. The filing ID is the AIF public presentation UUID; its numeric presentation ID is retained separately. Related-company rows are excluded. Individual and consolidated submissions remain separate. Some PDFs are scans and provide no text without OCR.


In [2]:
argentine_matches = search_companies("Central Puerto", market="AR", limit=3)
[(entity.entity_id, entity.name) for entity in argentine_matches]


[('33650305499', 'Central Puerto')]

In [3]:
argentina = Company(argentine_matches[0])
ar_report = argentina.filings(form="annual", limit=1).latest()
print(ar_report.filing_id, ar_report.provenance.source_id)
print(ar_report.period_end, ar_report.filing_date)
[(doc.filename, doc.document_type, doc.is_primary) for doc in ar_report.documents]


8376fa49-ac56-4787-ab6a-052e0fb7d9af 3491472
2025-12-31 2026-03-06


[('12-2025 - Central Puerto S.A. (Sin Reseña) Legalizado.pdf', 'ABEstadoContable', True), ('Memoria CPSA 2025_VF.pdf', 'ABMemoria', False), ('Informe EEFF Consolidados CPSA 2025 - Legalizado.pdf', 'ABInformeAuditorIndependiente', False), ('CPSA - COM. FISCALIZADORA - ACTA N 431 - EEFF 31.12.25.pdf', 'ABInformeComisionFiscalizadoraSindico', False), ('12-2025 - Central Puerto S.A. (Reseña Informativa).pdf', 'ABResenaInformativa', False)]

In [4]:
ar_original = ar_report.primary_document
ar_original.download(destination / "central-puerto-statements.pdf")
print("SHA-256:", ar_original.sha256())
print(ar_original.page(1).text[:220])


SHA-256: 3fb54eafd1dff23dbf02b44ba6a62c57f5aa7f4d39b652cec517efc00968a264
 
 
 
 
 
 
 
 
 
 
Central Puerto S.A. 
 
Estados Financieros Consolidados y Reseña Informativa correspondientes al 
ejercicio finalizado el 31 de diciembre de 202 5, juntamente con el Informe de los 
Auditores Independ


## Peru: selector ID, expediente, PDF and classic XBRL

SMV's numeric selector ID is not a RUC. Annual reports can include a full financial-statements PDF and a separate XBRL instance. `filing_date` comes from presentation metadata; the source year/quarter is not enough to invent an exact fiscal closing date. Discovery orders by report year, then presentation date within that year.


In [5]:
peruvian_matches = search_companies("Alicorp", market="PE", limit=3)
[(entity.entity_id, entity.name) for entity in peruvian_matches]


[('73', 'ALICORP S.A.A.')]

In [6]:
peru = Company(peruvian_matches[0])
pe_report = peru.filings(form="annual", limit=1).latest()
print(pe_report.filing_id, pe_report.provenance.source_id)
print("Presented:", pe_report.filing_date, "Exact period end:", pe_report.period_end)
[(doc.document_type, doc.media_type, doc.is_primary) for doc in pe_report.documents]


73:2025:A:C:2026015256 2026015256
Presented: 2026-03-31 Exact period end: None


[('Archivo Estructurado XBRL de Información Financiera', 'application/xml', False), ('Estados Financieros y Dictamen', 'application/pdf', True)]

In [7]:
pe_original = pe_report.primary_document
pe_original.download(destination / "alicorp-statements.pdf")
print(pe_original.page(1).text[:220])
facts = pe_report.facts
print("Raw fact occurrences:", len(facts))
print("Context date:", facts[0].period_end)


Alicorp S.A.A. y Subsidiarias 
Estados financieros consolidados al 31 de diciembre de 2025 y de 
2024, junto con el informe de los auditores independientes 
 
Raw fact occurrences: 1004
Context date: 2025-12-31


## Interim statements and notes are separate

`interim` and `quarterly` both select SMV's Intermedio category. The source's Roman quarter remains metadata. An original XBRL instance can be primary when the PDF contains only notes. Inspect formats before assuming a PDF report. Some source XML files declare UTF-8 but contain Windows-1252 bytes: parsing records the decoding fallback on fact provenance; downloaded originals remain unchanged.


In [8]:
interim = peru.filings(form="quarterly", limit=1).latest()
print("Source quarter:", interim.provider_metadata["source_quarter"])
[(doc.document_type, doc.media_type, doc.role, doc.is_primary) for doc in interim.documents]


Source quarter: II


[('Archivo Estructurado XBRL de Información Financiera', 'application/xml', 'data', True), ('Notas a los Estados Financieros', 'application/pdf', 'exhibit', False)]

In [9]:
print("Interim raw facts:", len(interim.facts))
print("Direct case lookup:", peru.get_filing(pe_report.filing_id).filing_id)
argentina.close()
peru.close()


Interim raw facts: 1184
Direct case lookup: 73:2025:A:C:2026015256


See the [Argentina guide](../../docs/markets/ar.md) and [Peru guide](../../docs/markets/pe.md) for bounds, source rights, dates, identifiers and known gaps. Neither route classifies amendments. Peru cannot apply filing-date windows to report-year discovery. Argentina buffers selected PDFs; Peru supports streamed saves.
